# Nauwkeurigheid zonnepaneel-voorspelling per bewolkingsniveau

In deze notebook vergelijken we de **voorspelde opbrengst** (gebaseerd op de GTI van Open-Meteo) met de **werkelijke opbrengst** van de panelen, beide uit je SQLite-database. Vervolgens onderzoeken we hoe de waarden voor **lage, middelbare en hoge bewolking** (Open-Meteo: `cloud_cover_low` / `cloud_cover_mid` / `cloud_cover_high`, waarden 0–100 %) de nauwkeurigheid beïnvloeden.

Aanpak:

1. Database en kolommen inspecteren
2. Data laden en opschonen
3. Verkennende analyse (EDA)
4. Fouten per bewolkingsniveau (MAE, RMSE, bias, MAPE)
5. **Leermodel** dat de werkelijke opbrengst voorspelt uit de ruwe voorspelling + bewolkingswaarden
6. Vergelijking: ruwe basisvoorspelling vs. gecorrigeerd model


In [ ]:
# Verwijder het # als packages ontbreken in je omgeving:
# %pip install pandas numpy scikit-learn matplotlib

import sqlite3
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.model_selection import TimeSeriesSplit
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

plt.rcParams["figure.figsize"] = (12, 4)
pd.options.display.float_format = "{:,.2f}".format


## 1. Configuratie

Pas hieronder het pad naar je SQLite-bestand en de SQL-query aan jouw database aan. **Belangrijk:** gebruik de aliassen zoals in het voorbeeld (`tijd`, `voorspeld`, `werkelijk`, `bewolking_laag`, `bewolking_middel`, `bewolking_hoog`, optioneel `gti`), want de rest van de notebook werkt met deze namen.

> Tip: Open-Meteo levert de bewolking per laag als `cloud_cover_low`, `cloud_cover_mid` en `cloud_cover_high` (oudere API-namen: `cloudcover_low` / `_mid` / `_high`), telkens als percentage 0–100 %. Heb je alleen totaalbewolking (`cloud_cover`) opgeslagen, laat dan de kolommen weg die je niet hebt.


In [ ]:
DB_PATH = "solar.db"

# Vul hier jouw tabel- en kolomnamen in; de aliassen (AS ...) niet veranderen.
QUERY = """
SELECT
    timestamp          AS tijd,
    forecast_power     AS voorspeld,
    actual_power       AS werkelijk,
    cloud_cover_low    AS bewolking_laag,
    cloud_cover_mid    AS bewolking_middel,
    cloud_cover_high   AS bewolking_hoog,
    gti
         AS gti
FROM metingen
ORDER BY tijd
"""

con = sqlite3.connect(DB_PATH)


## 2. Database inspecteren

Zo controleer je welke tabellen en kolommen er werkelijk zijn. Klopt de query in stap 1 niet, pas die dan aan en start de kernel opnieuw vanaf stap 3.


In [ ]:
tabellen = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table'", con)

for tabelnaam in tabellen["name"]:
    print(f"=== {tabelnaam} ===")
    kolommen = pd.read_sql(f"PRAGMA table_info('{tabelnaam}')", con)
    print(kolommen[["name", "type"]].to_string(index=False))
    print(pd.read_sql(f"SELECT * FROM '{tabelnaam}' LIMIT 3", con))
    print()


## 3. Data laden


In [ ]:
df = pd.read_sql(QUERY, con, parse_dates=["tijd"])
print(df.shape)
df.head()


## 4. Opschonen en afgeleide grootheden

- `residu` = werkelijk − voorspeld (positief = onderschatting van de opbrengst)
- `abs_pct` = absolute procentuele fout (alleen zinvol bij voldoende opbrengst)
- `dag` = de subset met zinvolle productie (> 5 % van het maximum), voor MAPE


In [ ]:
df = df.sort_values("tijd").dropna(subset=["voorspeld", "werkelijk"]).reset_index(drop=True)

df["residu"] = df["werkelijk"] - df["voorspeld"]
df["abs_pct"] = df["residu"].abs() / df["werkelijk"].clip(lower=1) * 100
df["uur"] = df["tijd"].dt.hour

drempel = 0.05 * df["werkelijk"].max()
dag = df[df["werkelijk"] >= drempel]

print(f"rijen: {len(df)}, waarvan overdag (productie >= {drempel:.1f}): {len(dag)}")
df.describe().T


## 5. Verkennende analyse


In [ ]:
df.plot(x="tijd", y=["voorspeld", "werkelijk"], linewidth=1, alpha=0.85)
plt.ylabel("opbrengst")
plt.title("voorspelling vs. werkelijkheid over de hele periode")
plt.show()


In [ ]:
fig, assen = plt.subplots(1, 2, figsize=(12, 4))

lim = max(df["voorspeld"].max(), df["werkelijk"].max()) * 1.05
assen[0].scatter(df["voorspeld"], df["werkelijk"], s=4, alpha=0.3)
assen[0].plot([0, lim], [0, lim], "r--", linewidth=1)
assen[0].set(xlabel="voorspeld", ylabel="werkelijk", title="voorspeld vs. werkelijk")

assen[1].hist(df["residu"], bins=60)
assen[1].axvline(0, color="r", linestyle="--")
assen[1].set(xlabel="residu (werkelijk − voorspeld)", title="verdeling van het residu")

plt.tight_layout()
plt.show()


## 6. Invloed van bewolkingsniveaus

Elke bewolkingsvariabele (laag, middel, hoog) wordt ingedeeld in drie klassen naar waarde: 0–33 % = **laag**, 33–67 % = **middel**, 67–100 % = **hoog**. Zo kunnen we per niveau zien of de voorspelling daar systematisch beter of slechter is.

- **MAE**: gemiddelde absolute fout (eenheid = opbrengst)
- **RMSE**: gemiddelde kwadratische fout, straft grote missers extra af
- **bias**: gemiddeld residu; negatief = de voorspelling zit structureel te hoog
- **MAPE %**: procentuele fout, alleen berekend op de `dag`-subset


In [ ]:
bins = [0, 33, 67, 100]
labels = ["laag", "middel", "hoog"]

bewolkingsvars = [v for v in ["bewolking_laag", "bewolking_middel", "bewolking_hoog"] if v in df.columns]

for var in bewolkingsvars:
    df[f"{var}_cat"] = pd.cut(df[var], bins=bins, labels=labels, include_lowest=True)

print("gevonden bewolkingsvariabelen:", bewolkingsvars)
df[[v for var in bewolkingsvars for v in (var, f"{var}_cat")]].head()


In [ ]:
def fouttabel(data, groep):
    tabel = data.groupby(groep, observed=True).agg(
        n=("residu", "size"),
        MAE=("residu", lambda s: s.abs().mean()),
        RMSE=("residu", lambda s: np.sqrt(np.mean(s**2))),
        bias=("residu", "mean"),
    )
    overdag = data[data["werkelijk"] >= drempel]
    tabel["MAPE_%"] = overdag.groupby(groep, observed=True)["abs_pct"].mean()
    return tabel.round(2)

for var in bewolkingsvars:
    print(f"=== {var} ===")
    display(fouttabel(df, f"{var}_cat"))
    print()


In [ ]:
for var in bewolkingsvars:
    tabel = fouttabel(df, f"{var}_cat")

    fig, assen = plt.subplots(1, 2, figsize=(11, 3.5))
    tabel["MAE"].plot.bar(ax=assen[0], color="tab:blue")
    assen[0].set(title=f"MAE per {var}", xlabel="")

    tabel["bias"].plot.bar(ax=assen[1], color="tab:orange")
    assen[1].axhline(0, color="k", linewidth=1)
    assen[1].set(title=f"bias per {var} (positief = onderschatting)", xlabel="")

    plt.tight_layout()
    plt.show()

for var in bewolkingsvars:
    df.boxplot(column="residu", by=f"{var}_cat", showfliers=False)
    plt.suptitle("")
    plt.title(f"residu (werkelijk − voorspeld) per {var}")
    plt.show()


## 7. Leermodel

Nu bouwen we een model dat **leert** hoe de voorspelling verbeterd kan worden. Het model (`HistGradientBoostingRegressor`) voorspelt de werkelijke opbrengst uit:

- de ruwe voorspelling zelf,
- de drie bewolkingswaarden (laag, middel, hoog),
- het uur van de dag (als sinus/cosinus, want de opbrengst is cyclisch).

Belangrijke keuzes:

- **Tijdsgebieden splitsen**: we trainen op de eerste 80 % van de tijd en testen op de laatste 20 %. Dat voorkomt dat het model "uit de toekomst leert".
- **Baseline**: de onveranderde voorspelling uit de database. Alleen als het model op de testset (en in tijdserie-kruisvalidatie) beter is, is de correctie waardevol.

*Alternatief: je kunt ook het **residu** als target nemen en het model als correctieterm op de ruwe voorspelling gebruiken — verander hiervoor alleen `y = df["residu"]` en tel de voorspelling erbij op.*


In [ ]:
df["uur_sin"] = np.sin(2 * np.pi * df["uur"] / 24)
df["uur_cos"] = np.cos(2 * np.pi * df["uur"] / 24)

kandidaten = ["voorspeld", "gti", "bewolking_laag", "bewolking_middel", "bewolking_hoog", "uur_sin", "uur_cos"]
kolommen_beschikbaar = [k for k in kandidaten if k in df.columns]
print("features:", kolommen_beschikbaar)

X = df[kolommen_beschikbaar]
y = df["werkelijk"]

split = int(len(df) * 0.8)
X_train, X_test = X.iloc[:split], X.iloc[split:]
y_train, y_test = y.iloc[:split], y.iloc[split:]


In [ ]:
model = HistGradientBoostingRegressor(
    loss="absolute_error",
    max_iter=300,
    learning_rate=0.05,
    random_state=42,
)
model.fit(X_train, y_train)

df["gecorrigeerd"] = np.nan
df.loc[df.index[split:], "gecorrigeerd"] = model.predict(X_test)
test = df.iloc[split:].copy()


In [ ]:
tscv = TimeSeriesSplit(n_splits=5)
scores_basis, scores_model = [], []

for train_idx, test_idx in tscv.split(X):
    m = HistGradientBoostingRegressor(loss="absolute_error", max_iter=300, learning_rate=0.05, random_state=42)
    m.fit(X.iloc[train_idx], y.iloc[train_idx])
    voorspelling = m.predict(X.iloc[test_idx])

    scores_basis.append(mean_absolute_error(y.iloc[test_idx], df["voorspeld"].iloc[test_idx]))
    scores_model.append(mean_absolute_error(y.iloc[test_idx], voorspelling))

overzicht = pd.DataFrame({"basisvoorspelling": scores_basis, "leermodel": scores_model})
print(overzicht.round(2))
print("\ngemiddelde MAE:")
print(overzicht.mean().round(2))
print(f"\nverbetering leermodel t.o.v. basis: {100 * (1 - overzicht['leermodel'].mean() / overzicht['basisvoorspelling'].mean()):.1f} %")


In [ ]:
def vergelijk(data, label=""):
    tabel = pd.DataFrame({
        "basis": (mean_absolute_error(data["werkelijk"], data["voorspeld"]),
                  np.sqrt(mean_squared_error(data["werkelijk"], data["voorspeld"])),
                  r2_score(data["werkelijk"], data["voorspeld"])),
        "model": (mean_absolute_error(data["werkelijk"], data["gecorrigeerd"]),
                  np.sqrt(mean_squared_error(data["werkelijk"], data["gecorrigeerd"])),
                  r2_score(data["werkelijk"], data["gecorrigeerd"])),
    }, index=["MAE", "RMSE", "R2"]).T.round(2)
    tabel.index.name = label
    return tabel

display(vergelijk_tabel := vergelijk(test, "algemeen (testset 20 %)"))

for var in bewolkingsvars:
    delen = [vergelijk(g, str(cat)) for cat, g in test.groupby(f"{var}_cat", observed=True)]
    display(pd.concat(delen).rename_axis(var))


In [ ]:
sub = X_test.iloc[:: max(1, len(X_test) // 5000)]
resultaat = permutation_importance(model, sub, y_test.loc[sub.index], n_repeats=10, random_state=42, scoring="neg_mean_absolute_error")

belang = pd.Series(resultaat.importances_mean, index=kolommen_beschikbaar).sort_values()
belang.plot.barh(color="tab:green")
plt.title("belang van elke feature (permutatie)")
plt.xlabel("toename van MAE wanneer de feature willekeurig wordt gemaakt")
plt.tight_layout()
plt.show()


In [ ]:
fig, assen = plt.subplots(2, 1, figsize=(13, 8))

stuk = test.tail(24 * 14)
stuk.plot(x="tijd", y=["werkelijk", "voorspeld", "gecorrigeerd"], linewidth=1, alpha=0.85, ax=assen[0])
assen[0].set(title="laatste twee weken van de testset", ylabel="opbrengst")

assen[1].scatter(test["werkelijk"], test["gecorrigeerd"], s=4, alpha=0.3)
lim = max(test["werkelijk"].max(), test["gecorrigeerd"].max()) * 1.05
assen[1].plot([0, lim], [0, lim], "r--", linewidth=1)
assen[1].set(xlabel="werkelijk", ylabel="model", title="model vs. werkelijk (testset)")

plt.tight_layout()
plt.show()


## 8. Wat kun je aflezen

- **Stap 6:** de tabellen en balkgrafieken tonen of de fout groeit bij hoge waarden van één van de drie bewolkingslagen, en of de fout systematisch is (bias) of alleen groter wordt (MAE/RMSE). Een grote RMSE t.o.v. de MAE wijst op incidentele grote missers onder bepaalde bewolkingsomstandigheden.
- **Stap 7:** is de gemiddelde MAE van het leermodel lager dan die van de basisvoorspelling, dan leert het model iets nuttigs uit de bewolkingswaarden. Het feature-belang laat zien welke bewolkingslaag de meeste bijdrage levert aan de correctie.
- **Let op:** train alleen op historische data,evalueer op een langere periode (minimaal enkele weken, idealiter over seizoenen heen), en genereer de bewolkingswaarden bij productiegebruik opnieuw via de API van Open-Meteo voor de toekomstige tijdstippen.

### Mogelijke vervolgstappen
- Meer features opnemen: buitentemperatuur, wind (koeling van de panelen), direct/diffuse straling, seizoen.
- Aparte modellen trainen per seizoen of per bewolkingscombinatie.
- Het `residu` als target gebruiken en het model als plus-correctie op de ruwe GTI-voorspelling.
